In [ ]:
import tkinter as tk
import random
import time
from collections import deque

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
TILE = 90
GAP = 5
ANIM = 400

OBS_INDICES = (0, 1, 2)

def observe(board):
    return tuple(board[i] for i in OBS_INDICES)

def misplaced(board):
    return sum(1 for i, val in enumerate(board) if val != 0 and val != GOAL[i])

def apply_action(board, action):
    idx = board.index(0)
    r, c = divmod(idx, 3)
    dr, dc = action
    nr, nc = r + dr, c + dc
    if 0 <= nr < 3 and 0 <= nc < 3:
        ni = nr * 3 + nc
        temp = list(board)
        temp[idx], temp[ni] = temp[ni], temp[idx]
        return tuple(temp)
    return None

ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
ACTION_NAMES = {(-1, 0): "UP", (1, 0): "DOWN", (0, -1): "LEFT", (0, 1): "RIGHT"}

def get_all_reachable_boards():
    visited = {GOAL}
    queue = deque([GOAL])
    while queue:
        b = queue.popleft()
        for action in ACTIONS:
            nb = apply_action(b, action)
            if nb and nb not in visited:
                visited.add(nb)
                queue.append(nb)
    return visited

ALL_BOARDS = None

def get_all_boards():
    global ALL_BOARDS
    if ALL_BOARDS is None:
        ALL_BOARDS = get_all_reachable_boards()
    return ALL_BOARDS

def initial_belief(board):
    obs = observe(board)
    all_boards = get_all_boards()
    return frozenset(b for b in all_boards if observe(b) == obs)

def update_belief(belief, action, observation):
    new_belief = set()
    for board in belief:
        nb = apply_action(board, action)
        if nb is None:
            nb = board  
        if observe(nb) == observation:
            new_belief.add(nb)
    return frozenset(new_belief)

def partially_observable_bfs(true_board):
    start_obs = observe(true_board)
    start_belief = initial_belief(true_board)
    start = (true_board, start_belief)
    queue = deque()
    queue.append((true_board, start_belief, [true_board], [start_belief]))
    visited = {start}
    nodes = 0

    while queue:
        cur_board, cur_belief, path, bpath = queue.popleft()
        nodes += 1
        if cur_board == GOAL:
            return path, bpath, nodes
        for action in ACTIONS:
            nb = apply_action(cur_board, action)
            if nb is None:
                continue  
            obs = observe(nb)
            new_belief = update_belief(cur_belief, action, obs)
            if not new_belief:
                continue
            key = (nb, new_belief)
            if key in visited:
                continue
            visited.add(key)
            queue.append((nb, new_belief, path + [nb], bpath + [new_belief]))
            if nodes > 100000:
                return path, bpath, nodes
    return [], [], nodes


def is_solvable(board):
    arr = [x for x in board if x != 0]
    inv = sum(
        1 for i in range(len(arr))
        for j in range(i + 1, len(arr))
        if arr[i] > arr[j]
    )
    return inv % 2 == 0

def random_board():
    b = list(GOAL)
    while True:
        random.shuffle(b)
        t = tuple(b)
        if is_solvable(t) and t != GOAL:
            return t


class App:
    def __init__(self, root):
        self.root = root
        self.root.title("8 Puzzle — Partially Observable BFS")
        self.root.configure(bg="white")
        self.root.resizable(False, False)
        self.root.after(100, self._precompute)
        self.board = random_board()
        self.path = []
        self.belief_path = []
        self.moves = []
        self.step = 0
        self.playing = False

        main = tk.Frame(root, bg="white")
        main.pack(padx=10, pady=10)

        left = tk.Frame(main, bg="white")
        left.pack(side="left", padx=10)

        self.canvas = tk.Canvas(
            left,
            width=3 * TILE + 4 * GAP,
            height=3 * TILE + 4 * GAP,
            bg="white", highlightthickness=0
        )
        self.canvas.pack()

        self.step_label = tk.Label(
            left, text="Steps:", font=("Arial", 12), bg="white"
        )
        self.step_label.pack(pady=6)

        self.lb_heuristic = tk.Label(
            left, text="h(n) = —",
            bg="white", fg="#c05000", font=("Arial", 10)
        )
        self.lb_heuristic.pack()
        obs_frame = tk.Frame(left, bg="white", bd=1, relief="groove")
        obs_frame.pack(pady=4, fill="x", padx=4)
        tk.Label(obs_frame, text="Observation (hàng 1):",
                 bg="white", font=("Arial", 9, "bold")).pack(side="left", padx=4)
        self.lb_obs = tk.Label(obs_frame, text="[?, ?, ?]",
                                bg="white", fg="#0055cc", font=("Consolas", 10))
        self.lb_obs.pack(side="left", padx=4)

        info = tk.Frame(left, bg="white")
        info.pack(pady=4)
        self.lb_nodes = tk.Label(info, text="Nodes: 0", bg="white")
        self.lb_nodes.grid(row=0, column=0, padx=6)
        self.lb_steps = tk.Label(info, text="Steps: 0", bg="white")
        self.lb_steps.grid(row=0, column=1, padx=6)
        self.lb_time = tk.Label(info, text="Time: 0 ms", bg="white")
        self.lb_time.grid(row=0, column=2, padx=6)

        info2 = tk.Frame(left, bg="white")
        info2.pack(pady=2)
        self.lb_belief = tk.Label(
            info2, text="Belief size: —", bg="white", fg="#006400", font=("Arial", 10)
        )
        self.lb_belief.grid(row=0, column=0, padx=6)
        self.lb_visited = tk.Label(
            info2, text="Visited: 0", bg="white", fg="#00008b", font=("Arial", 10)
        )
        self.lb_visited.grid(row=0, column=1, padx=6)

        btns = tk.Frame(left, bg="white")
        btns.pack(pady=6)
        tk.Button(btns, text="Shuffle", width=10, command=self.shuffle).grid(row=0, column=0, padx=4)
        tk.Button(btns, text="Reset",   width=10, command=self.reset  ).grid(row=0, column=1, padx=4)
        tk.Button(btns, text="Solve",   width=10, command=self.solve  ).grid(row=0, column=2, padx=4)
        tk.Button(btns, text="Step",    width=34, command=self.next_step).grid(
            row=1, column=0, columnspan=3, pady=4
        )

        right = tk.Frame(main, bg="white")
        right.pack(side="right", padx=10)

        tk.Label(
            right,
            text="Solution (Partially Observable BFS)",
            font=("Arial", 13, "bold"),
            bg="white"
        ).pack()

        self.solution_box = tk.Text(right, width=28, height=25, font=("Consolas", 10))
        self.solution_box.pack()

        self.draw()

    def _precompute(self):
        self.step_label.config(text="Đang khởi tạo...")
        get_all_boards()
        self.step_label.config(text="Sẵn sàng")

    def current_board(self):
        if self.path and self.step < len(self.path):
            return self.path[self.step]
        return self.board

    def current_belief(self):
        if self.belief_path and self.step < len(self.belief_path):
            return self.belief_path[self.step]
        return None

    def draw(self):
        self.canvas.delete("all")
        board = self.current_board()
        belief = self.current_belief()
        h_val = misplaced(board)
        obs = observe(board)

        for i, val in enumerate(board):
            r, c = divmod(i, 3)
            x1 = GAP + c * (TILE + GAP)
            y1 = GAP + r * (TILE + GAP)
            x2 = x1 + TILE
            y2 = y1 + TILE

            is_observed = i in OBS_INDICES

            if val == 0:
                self.canvas.create_rectangle(
                    x1, y1, x2, y2, fill="#e8f5e9", outline="#aaa",
                    width=2, dash=(4, 3)
                )
            else:
                wrong = (val != GOAL[i])
                if is_observed:
                    fill = "#e0f0ff" if not wrong else "#ffe0e0"
                    outline_color = "#0055cc"
                    outline_w = 3
                else:
                    fill = "#f5f5f5" if not wrong else "#ffe0e0"
                    outline_color = "#999999"
                    outline_w = 2

                self.canvas.create_rectangle(
                    x1, y1, x2, y2, fill=fill,
                    outline=outline_color, width=outline_w
                )
                self.canvas.create_text(
                    (x1 + x2) // 2, (y1 + y2) // 2,
                    text=str(val), font=("Arial", 28),
                    fill="red" if wrong else ("black")
                )
            if not self.path and not is_observed and val != 0:
                self.canvas.create_text(
                    x2 - 12, y1 + 12,
                    text="?", font=("Arial", 10, "bold"), fill="#aaa"
                )

        self.lb_heuristic.config(text=f"h(n) = {h_val}")
        self.lb_obs.config(text=str(list(obs)))
        if belief is not None:
            self.lb_belief.config(text=f"Belief size: {len(belief)}")

    def shuffle(self):
        self.playing = False
        self.board = random_board()
        self.path = []
        self.belief_path = []
        self.moves = []
        self.step = 0
        self._reset_labels()
        self.draw()

    def reset(self):
        self.playing = False
        self.board = GOAL
        self.path = []
        self.belief_path = []
        self.moves = []
        self.step = 0
        self._reset_labels()
        self.draw()

    def _reset_labels(self):
        self.lb_nodes.config(text="Nodes: 0")
        self.lb_steps.config(text="Steps: 0")
        self.lb_time.config(text="Time: 0 ms")
        self.step_label.config(text="Steps:")
        self.lb_heuristic.config(text="h(n) = —")
        self.lb_belief.config(text="Belief size: —")
        self.lb_visited.config(text="Visited: 0")
        self.lb_obs.config(text="[?, ?, ?]")
        self.solution_box.delete("1.0", tk.END)

    def get_direction(self, old_state, new_state):
        diff = new_state.index(0) - old_state.index(0)
        return {-3: "UP", 3: "DOWN", -1: "LEFT", 1: "RIGHT"}.get(diff, "?")

    def solve(self):
        if self.playing:
            return
        get_all_boards()

        t0 = time.perf_counter()
        path, belief_path, nodes = partially_observable_bfs(self.board)
        ms = (time.perf_counter() - t0) * 1000

        self.path = path
        self.belief_path = belief_path
        self.moves = [
            self.get_direction(path[i], path[i + 1])
            for i in range(len(path) - 1)
        ] if len(path) > 1 else []
        self.step = 0

        self.lb_nodes.config(text=f"Nodes: {nodes}")
        self.lb_steps.config(text=f"Steps: {len(path) - 1 if path else 0}")
        self.lb_time.config(text=f"Time: {ms:.2f} ms")
        self.lb_visited.config(text=f"Visited: {nodes}")

        solved = bool(path) and path[-1] == GOAL
        self.step_label.config(text="✓ SOLVED" if solved else "✗ Not solved")

        self.solution_box.delete("1.0", tk.END)
        if path:
            self.playing = True
            self.animate()

    def _update_solution_box(self):
        if not self.path or self.step == 0:
            return
        self.solution_box.config(state=tk.NORMAL)
        self.solution_box.delete("1.0", tk.END)
        for i in range(1, self.step + 1):
            m = self.moves[i - 1] if i - 1 < len(self.moves) else "?"
            h = misplaced(self.path[i])
            belief = self.belief_path[i] if i < len(self.belief_path) else set()
            obs = observe(self.path[i])
            self.solution_box.insert(
                tk.END,
                f"{i:2}. {m:<5} h={h} |B|={len(belief)}\n"
                f"    obs={list(obs)}\n"
            )
        self.solution_box.tag_remove("highlight", "1.0", tk.END)
        line = self.step * 2
        self.solution_box.tag_add("highlight", f"{line}.0", f"{line+1}.end")
        self.solution_box.tag_config("highlight", background="#cce5ff")
        self.solution_box.see(tk.END)

    def animate(self):
        if not self.playing:
            return
        self.draw()
        if self.step == 0:
            self.step_label.config(text="START")
        else:
            m = self.moves[self.step - 1] if self.step - 1 < len(self.moves) else "?"
            self.step_label.config(text=f"Step {self.step}: {m}")

        self._update_solution_box()

        if self.step < len(self.path) - 1:
            self.step += 1
            self.root.after(ANIM, self.animate)
        else:
            self.playing = False
            solved = self.path[-1] == GOAL
            self.step_label.config(text="✓ SOLVED!" if solved else "✗ Stuck")

    def next_step(self):
        self.playing = False
        if not self.path:
            return
        if self.step < len(self.path) - 1:
            self.step += 1
            self.draw()
            m = self.moves[self.step - 1] if self.step - 1 < len(self.moves) else "?"
            self.step_label.config(text=f"Step {self.step}: {m}")
            self._update_solution_box()


if __name__ == "__main__":
    root = tk.Tk()
    app = App(root)
    root.mainloop()